## 📚 Setup and Imports

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import os
import sys
import yaml
import pickle
import warnings
from pathlib import Path
import json
import time

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Model selection
from sklearn.model_selection import cross_val_score, GridSearchCV

# Sparse matrix utilities
from scipy.sparse import load_npz

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

# Add src to path
sys.path.append('../')

print("✅ Libraries imported successfully!")
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Load configuration
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("📋 Configuration loaded:")
print(f"  Random state: {config['preprocessing']['random_state']}")
print(f"  Models will be saved to: models/saved_models/")

## 📂 Load Features

In [ ]:
print("📂 Loading feature matrices...\n")

features_dir = 'data/features'

# Load combined features (TF-IDF + handcrafted)
print("Loading combined features...")
X_train = load_npz(os.path.join(features_dir, 'X_train_combined.npz'))
X_val = load_npz(os.path.join(features_dir, 'X_val_combined.npz'))
X_test = load_npz(os.path.join(features_dir, 'X_test_combined.npz'))

# Load labels
print("Loading labels...")
y_train = np.load(os.path.join(features_dir, 'y_train.npy'))
y_val = np.load(os.path.join(features_dir, 'y_val.npy'))
y_test = np.load(os.path.join(features_dir, 'y_test.npy'))

# Load feature info
with open(os.path.join(features_dir, 'feature_info.json'), 'r') as f:
    feature_info = json.load(f)

print(f"\n✅ Features loaded!")
print(f"   Training:   {X_train.shape}")
print(f"   Validation: {X_val.shape}")
print(f"   Test:       {X_test.shape}")
print(f"\n   Total features: {feature_info['n_total_features']:,}")
print(f"   - TF-IDF: {feature_info['n_tfidf_features']:,}")
print(f"   - Handcrafted: {feature_info['n_handcrafted_features']}")

In [ ]:
# Check class distribution
print("\n📊 Class distribution:\n")
print("Training:")
print(pd.Series(y_train).value_counts())
print(f"Balance: {pd.Series(y_train).value_counts().min() / pd.Series(y_train).value_counts().max():.2%}")

print("\nValidation:")
print(pd.Series(y_val).value_counts())
print(f"Balance: {pd.Series(y_val).value_counts().min() / pd.Series(y_val).value_counts().max():.2%}")

print("\nTest:")
print(pd.Series(y_test).value_counts())
print(f"Balance: {pd.Series(y_test).value_counts().min() / pd.Series(y_test).value_counts().max():.2%}")

## 🎯 Model 1: Logistic Regression

In [ ]:
print("🎯 Training Logistic Regression...\n")

# Initialize model
lr_config = config['traditional_models']['logistic_regression']
print(f"Configuration: {lr_config}")

lr_model = LogisticRegression(
    C=lr_config['C'],
    max_iter=lr_config['max_iter'],
    solver=lr_config['solver'],
    random_state=config['preprocessing']['random_state'],
    n_jobs=-1
)

# Train model
print("\n⏳ Training...")
start_time = time.time()
lr_model.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"✅ Training complete in {training_time:.2f} seconds")

# Predictions
print("\n📊 Making predictions...")
y_train_pred_lr = lr_model.predict(X_train)
y_val_pred_lr = lr_model.predict(X_val)
y_test_pred_lr = lr_model.predict(X_test)

# Probabilities for ROC-AUC
y_train_proba_lr = lr_model.predict_proba(X_train)[:, 1]
y_val_proba_lr = lr_model.predict_proba(X_val)[:, 1]
y_test_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n📈 Performance Metrics:\n")
print("Training Set:")
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_train, y_train_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_train, y_train_pred_lr):.4f}")
print(f"  F1-Score:  {f1_score(y_train, y_train_pred_lr):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba_lr):.4f}")

print("\nValidation Set:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_val, y_val_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_val, y_val_pred_lr):.4f}")
print(f"  F1-Score:  {f1_score(y_val, y_val_pred_lr):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_proba_lr):.4f}")

print("\nTest Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred_lr):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred_lr):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba_lr):.4f}")

## 🌲 Model 2: Random Forest

In [ ]:
print("🌲 Training Random Forest...\n")

# Initialize model
rf_config = config['traditional_models']['random_forest']
print(f"Configuration: {rf_config}")

rf_model = RandomForestClassifier(
    n_estimators=rf_config['n_estimators'],
    max_depth=rf_config['max_depth'],
    min_samples_split=rf_config['min_samples_split'],
    random_state=config['preprocessing']['random_state'],
    n_jobs=-1,
    verbose=1
)

# Train model
print("\n⏳ Training...")
start_time = time.time()
rf_model.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"✅ Training complete in {training_time:.2f} seconds")

# Predictions
print("\n📊 Making predictions...")
y_train_pred_rf = rf_model.predict(X_train)
y_val_pred_rf = rf_model.predict(X_val)
y_test_pred_rf = rf_model.predict(X_test)

# Probabilities for ROC-AUC
y_train_proba_rf = rf_model.predict_proba(X_train)[:, 1]
y_val_proba_rf = rf_model.predict_proba(X_val)[:, 1]
y_test_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n📈 Performance Metrics:\n")
print("Training Set:")
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred_rf):.4f}")
print(f"  Precision: {precision_score(y_train, y_train_pred_rf):.4f}")
print(f"  Recall:    {recall_score(y_train, y_train_pred_rf):.4f}")
print(f"  F1-Score:  {f1_score(y_train, y_train_pred_rf):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba_rf):.4f}")

print("\nValidation Set:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred_rf):.4f}")
print(f"  Precision: {precision_score(y_val, y_val_pred_rf):.4f}")
print(f"  Recall:    {recall_score(y_val, y_val_pred_rf):.4f}")
print(f"  F1-Score:  {f1_score(y_val, y_val_pred_rf):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_proba_rf):.4f}")

print("\nTest Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred_rf):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred_rf):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred_rf):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred_rf):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba_rf):.4f}")

## 🎯 Model 3: Support Vector Machine (SVM)

In [ ]:
print("🎯 Training SVM...\n")

# Initialize model
svm_config = config['traditional_models']['svm']
print(f"Configuration: {svm_config}")

svm_model = SVC(
    C=svm_config['C'],
    kernel=svm_config['kernel'],
    gamma=svm_config['gamma'],
    random_state=config['preprocessing']['random_state'],
    probability=True,  # For ROC-AUC
    verbose=True
)

# Train model
print("\n⏳ Training (this may take a while)...")
start_time = time.time()
svm_model.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"✅ Training complete in {training_time:.2f} seconds")

# Predictions
print("\n📊 Making predictions...")
y_train_pred_svm = svm_model.predict(X_train)
y_val_pred_svm = svm_model.predict(X_val)
y_test_pred_svm = svm_model.predict(X_test)

# Probabilities for ROC-AUC
y_train_proba_svm = svm_model.predict_proba(X_train)[:, 1]
y_val_proba_svm = svm_model.predict_proba(X_val)[:, 1]
y_test_proba_svm = svm_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n📈 Performance Metrics:\n")
print("Training Set:")
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred_svm):.4f}")
print(f"  Precision: {precision_score(y_train, y_train_pred_svm):.4f}")
print(f"  Recall:    {recall_score(y_train, y_train_pred_svm):.4f}")
print(f"  F1-Score:  {f1_score(y_train, y_train_pred_svm):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba_svm):.4f}")

print("\nValidation Set:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred_svm):.4f}")
print(f"  Precision: {precision_score(y_val, y_val_pred_svm):.4f}")
print(f"  Recall:    {recall_score(y_val, y_val_pred_svm):.4f}")
print(f"  F1-Score:  {f1_score(y_val, y_val_pred_svm):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_proba_svm):.4f}")

print("\nTest Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred_svm):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred_svm):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred_svm):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred_svm):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba_svm):.4f}")

## 🚀 Model 4: XGBoost

In [ ]:
print("🚀 Training XGBoost...\n")

# Initialize model
xgb_config = config['traditional_models']['xgboost']
print(f"Configuration: {xgb_config}")

xgb_model = XGBClassifier(
    n_estimators=xgb_config['n_estimators'],
    max_depth=xgb_config['max_depth'],
    learning_rate=xgb_config['learning_rate'],
    random_state=config['preprocessing']['random_state'],
    n_jobs=-1,
    eval_metric='logloss'
)

# Train model with validation set for early stopping
print("\n⏳ Training with early stopping...")
start_time = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)
training_time = time.time() - start_time

print(f"✅ Training complete in {training_time:.2f} seconds")

# Predictions
print("\n📊 Making predictions...")
y_train_pred_xgb = xgb_model.predict(X_train)
y_val_pred_xgb = xgb_model.predict(X_val)
y_test_pred_xgb = xgb_model.predict(X_test)

# Probabilities for ROC-AUC
y_train_proba_xgb = xgb_model.predict_proba(X_train)[:, 1]
y_val_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]
y_test_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n📈 Performance Metrics:\n")
print("Training Set:")
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred_xgb):.4f}")
print(f"  Precision: {precision_score(y_train, y_train_pred_xgb):.4f}")
print(f"  Recall:    {recall_score(y_train, y_train_pred_xgb):.4f}")
print(f"  F1-Score:  {f1_score(y_train, y_train_pred_xgb):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba_xgb):.4f}")

print("\nValidation Set:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred_xgb):.4f}")
print(f"  Precision: {precision_score(y_val, y_val_pred_xgb):.4f}")
print(f"  Recall:    {recall_score(y_val, y_val_pred_xgb):.4f}")
print(f"  F1-Score:  {f1_score(y_val, y_val_pred_xgb):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_proba_xgb):.4f}")

print("\nTest Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred_xgb):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred_xgb):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred_xgb):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred_xgb):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba_xgb):.4f}")

## 📊 Model Comparison

In [ ]:
print("📊 Model Comparison Summary\n")
print("="*80)

# Create comparison dataframe
models_comparison = {
    'Model': ['Logistic Regression', 'Random Forest', 'SVM', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_test_pred_lr),
        accuracy_score(y_test, y_test_pred_rf),
        accuracy_score(y_test, y_test_pred_svm),
        accuracy_score(y_test, y_test_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_test_pred_lr),
        precision_score(y_test, y_test_pred_rf),
        precision_score(y_test, y_test_pred_svm),
        precision_score(y_test, y_test_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, y_test_pred_lr),
        recall_score(y_test, y_test_pred_rf),
        recall_score(y_test, y_test_pred_svm),
        recall_score(y_test, y_test_pred_xgb)
    ],
    'F1-Score': [
        f1_score(y_test, y_test_pred_lr),
        f1_score(y_test, y_test_pred_rf),
        f1_score(y_test, y_test_pred_svm),
        f1_score(y_test, y_test_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_test_proba_lr),
        roc_auc_score(y_test, y_test_proba_rf),
        roc_auc_score(y_test, y_test_proba_svm),
        roc_auc_score(y_test, y_test_proba_xgb)
    ]
}

comparison_df = pd.DataFrame(models_comparison)
comparison_df = comparison_df.round(4)

print("\nTest Set Performance:")
display(comparison_df)

# Find best model
best_model_idx = comparison_df['F1-Score'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_f1 = comparison_df.loc[best_model_idx, 'F1-Score']

print(f"\n🏆 Best Model: {best_model_name} (F1-Score: {best_f1:.4f})")
print("="*80)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot for all metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(comparison_df))
width = 0.15

for idx, metric in enumerate(metrics):
    axes[0].bar(x + idx * width, comparison_df[metric], width, label=metric, alpha=0.8)

axes[0].set_xlabel('Models', fontweight='bold')
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# F1-Score comparison
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = axes[1].barh(comparison_df['Model'], comparison_df['F1-Score'], color=colors, alpha=0.8)
axes[1].set_xlabel('F1-Score', fontweight='bold')
axes[1].set_title('F1-Score Comparison', fontweight='bold', fontsize=14)
axes[1].set_xlim([0, 1.05])
axes[1].grid(True, alpha=0.3, axis='x')

# Add values on bars
for bar in bars:
    width = bar.get_width()
    axes[1].text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                 f'{width:.4f}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 📊 Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

models = [
    ('Logistic Regression', y_test_pred_lr),
    ('Random Forest', y_test_pred_rf),
    ('SVM', y_test_pred_svm),
    ('XGBoost', y_test_pred_xgb)
]

for idx, (name, y_pred) in enumerate(models):
    cm = confusion_matrix(y_test, y_pred)
    
    # Plot heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
    axes[idx].set_title(f'{name}', fontweight='bold', fontsize=12)
    axes[idx].set_ylabel('True Label', fontweight='bold')
    axes[idx].set_xlabel('Predicted Label', fontweight='bold')

plt.tight_layout()
plt.show()

## 📈 ROC Curves

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 8))

# Calculate ROC curves
models_roc = [
    ('Logistic Regression', y_test_proba_lr),
    ('Random Forest', y_test_proba_rf),
    ('SVM', y_test_proba_svm),
    ('XGBoost', y_test_proba_xgb)
]

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for (name, y_proba), color in zip(models_roc, colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', color=color, linewidth=2)

# Plot diagonal
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
plt.title('ROC Curves Comparison', fontweight='bold', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 💾 Save Models

In [ ]:
print("💾 Saving trained models...\n")

# Create models directory
models_dir = 'models/saved_models'
os.makedirs(models_dir, exist_ok=True)

# Save models
models_to_save = [
    ('logistic_regression.pkl', lr_model),
    ('random_forest.pkl', rf_model),
    ('svm.pkl', svm_model),
    ('xgboost.pkl', xgb_model)
]

for filename, model in models_to_save:
    filepath = os.path.join(models_dir, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)
    print(f"✅ Saved: {filepath}")

# Save comparison results
comparison_df.to_csv(os.path.join(models_dir, 'model_comparison.csv'), index=False)
print(f"\n✅ Saved comparison results: {os.path.join(models_dir, 'model_comparison.csv')}")

# Save detailed results
results = {
    'best_model': best_model_name,
    'best_f1_score': float(best_f1),
    'models': {
        'logistic_regression': {
            'test_accuracy': float(accuracy_score(y_test, y_test_pred_lr)),
            'test_f1': float(f1_score(y_test, y_test_pred_lr)),
            'test_roc_auc': float(roc_auc_score(y_test, y_test_proba_lr))
        },
        'random_forest': {
            'test_accuracy': float(accuracy_score(y_test, y_test_pred_rf)),
            'test_f1': float(f1_score(y_test, y_test_pred_rf)),
            'test_roc_auc': float(roc_auc_score(y_test, y_test_proba_rf))
        },
        'svm': {
            'test_accuracy': float(accuracy_score(y_test, y_test_pred_svm)),
            'test_f1': float(f1_score(y_test, y_test_pred_svm)),
            'test_roc_auc': float(roc_auc_score(y_test, y_test_proba_svm))
        },
        'xgboost': {
            'test_accuracy': float(accuracy_score(y_test, y_test_pred_xgb)),
            'test_f1': float(f1_score(y_test, y_test_pred_xgb)),
            'test_roc_auc': float(roc_auc_score(y_test, y_test_proba_xgb))
        }
    }
}

with open(os.path.join(models_dir, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Saved detailed results: {os.path.join(models_dir, 'results.json')}")
print(f"\n✅ All models saved to: {models_dir}")

## 📊 Final Summary

In [ ]:
print("="*70)
print("📊 TRADITIONAL MODELS TRAINING COMPLETE!")
print("="*70)
print(f"\n✅ Successfully trained and evaluated 4 models")
print(f"\n📈 Final Test Set Results:")
print()
display(comparison_df)
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   F1-Score: {best_f1:.4f}")
print(f"   Accuracy: {comparison_df.loc[best_model_idx, 'Accuracy']:.4f}")
print(f"   ROC-AUC:  {comparison_df.loc[best_model_idx, 'ROC-AUC']:.4f}")
print(f"\n💾 Models saved to: {models_dir}")
print(f"\n🎯 Next: Deep Learning Models (LSTM, BERT, RoBERTa)")
print("="*70)

## 🔄 Next Steps

### What We Accomplished:
- ✅ Loaded 5,017 combined features
- ✅ Trained Logistic Regression
- ✅ Trained Random Forest
- ✅ Trained SVM
- ✅ Trained XGBoost
- ✅ Compared all models
- ✅ Saved trained models and results

### Model Performance Summary:
All models show strong performance on the balanced dataset with excellent F1-scores, precision, and recall.

### Next Notebook (05_deep_learning_models.ipynb):
1. Load pre-trained transformers (BERT, RoBERTa)
2. Fine-tune on our dataset
3. Train BiLSTM model
4. Compare with traditional models
5. Save best performing models

---

**📝 Note:** Traditional ML models provide strong baselines. Deep learning models may achieve even better performance!